<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 05 · Native Kafka and Delta</h1><p>ENGINE_NOT_EXECUTED · Native lab draft. No execution outputs are supplied. Missing requirements stop the notebook; it never switches to a mock engine.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اللاب 05 · Kafka وDelta الأصليان</h1><p>ENGINE_NOT_EXECUTED · مسودة لاب أصلي دون مخرجات تنفيذ. توقف المتطلبات المفقودة الدفتر؛ لا ينتقل إلى محرك وهمي.</p></td></tr></tbody>
</table>

<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><p><a href="../../day04/README.md">Day 4</a> · <a href="../../day04/SOURCES.md">Sources</a></p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><p><a href="../../day04/README.md">اليوم الرابع</a> · <a href="../../day04/SOURCES.md">المصادر</a></p></td></tr></tbody>
</table>



<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Locate the existing project</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>1. تحديد المشروع الموجود</h2></td></tr></tbody>
</table>



In [ ]:
from pathlib import Path
import sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete course repository")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.workspace import require_fixed_dataset
require_fixed_dataset(SOURCE)
print("MASAR_SMALL_V1: original source hashes verified")

<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>2. Inspect requirements without claiming execution</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>2. فحص المتطلبات دون ادعاء التنفيذ</h2></td></tr></tbody>
</table>



In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("masar_day04_runner", ROOT / "scripts/run_day04.py")
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
readiness = runner.preflight("streaming")
print(json.dumps(readiness, indent=2))

<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>3. Fail explicitly when the environment is missing</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>3. التوقف الصريح عند نقص البيئة</h2></td></tr></tbody>
</table>



In [ ]:
from masar.runtime import EnvironmentUnavailable, require_environment
if readiness["issues"]:
    raise EnvironmentUnavailable("; ".join(readiness["issues"]))
require_environment()
print("Dependencies present. Native execution has not started yet.")

<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>4. Execute against real predecessor outputs</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>4. التنفيذ على مخرجات سابقة فعلية</h2></td></tr></tbody>
</table>



In [ ]:
from masar.runtime import start_spark
from masar.workspace import completed_bronze_workspace, workspace_path, write_json
from masar.streaming import run_stream_lab
work = completed_bronze_workspace(ROOT)
spark = None
try:
    spark = start_spark(work, kafka=True)
    result = run_stream_lab(spark, SOURCE, work)
except Exception as exc:
    write_json(workspace_path(work, "reports/day04_streaming_notebook_failure.json"), {
        "status": "FAILED", "complete_lab_success": False,
        "error_type": type(exc).__name__, "message": str(exc), "engine_started": spark is not None})
    raise
finally:
    if spark is not None:
        spark.stop()
print(json.dumps(result, ensure_ascii=False, indent=2))

<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>5. Check the actual evidence</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>5. فحص الأدلة الفعلية</h2></td></tr></tbody>
</table>



In [ ]:
assert result["engine_executed"] is True
assert result["checks"] and all(value is True for value in result["checks"].values())
print("Observed checks:", len(result["checks"]))
print("Run identity:", result["run_id"])
print("Record the generated paths in the lab notes. Keep raw artifacts out of Git.")